In [1]:

from pyspark.sql import functions as F

from atlas.common.config.loader import get_settings
from atlas.common.paths.loader import get_paths
from atlas.common.spark.session import get_spark_session

settings = get_settings("configs/base.yaml", "configs/local.yaml", "pyproject.toml")

In [2]:
spark = get_spark_session(settings.spark, settings.storage, settings.application.name)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/01 05:15:23 WARN Utils: Your hostname, Saileshs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.8 instead (on interface en0)
26/09/01 05:15:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/saileshpola/Desktop/AtlasProject/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/saileshpola/.ivy2.5.2/cache
The jars for the packages stored in: /Users/saileshpola/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1a8ecd1b-d70a-46f4-8147-72b77d6cf898;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.del

In [3]:
paths = get_paths(settings)

In [4]:
bronze_customer_path = paths.bronze_path("customer/cdc/customers/job")

In [5]:
customer_bronze_data = spark.read.format("parquet").load(bronze_customer_path)

In [6]:

customer_bronze_data.select(F.col("raw_key"), F.col("raw_value")).show(vertical=True, truncate=False, n=1)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [7]:
from pyspark.sql.types import LongType, StringType, StructField, StructType

customer_record_schema = StructType([
    StructField("customer_id", LongType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True),
    StructField("date_of_birth", LongType(), True),
    StructField("status", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("created_at", StringType(), False),
    StructField("updated_at", StringType(), False),
])

In [8]:
customer_source_schema = StructType([
    StructField("version", StringType(), False),
    StructField("connector", StringType(), False),
    StructField("name", StringType(), False),
    StructField("ts_ms", LongType(), False),
    StructField("snapshot", StringType(), True),
    StructField("db", StringType(), False),
    StructField("sequence", StringType(), True),
    StructField("ts_us", LongType(), True),
    StructField("ts_ns", LongType(), True),
    StructField("schema", StringType(), False),
    StructField("table", StringType(), False),
    StructField("txId", LongType(), True),
    StructField("lsn", LongType(), True),
    StructField("xmin", LongType(), True),
])

In [9]:
customer_payload_schema = StructType([
    StructField("before", customer_record_schema, True),
    StructField("after", customer_record_schema, True),
    StructField("source", customer_source_schema, False),
    StructField("op", StringType(), False),
    StructField("ts_ms", LongType(), False),
    StructField("ts_us", LongType(), True),
    StructField("ts_ns", LongType(), True),
])

In [10]:
customer_debezium_schema = StructType([
    StructField("payload", customer_payload_schema, True),
])

In [11]:
customer_parsed_data = customer_bronze_data.withColumn("debezium", F.from_json(F.col("raw_value")
                                                                               , customer_debezium_schema))

In [12]:
customer_parsed_data.select(
    "debezium.payload.before",
    "debezium.payload.after",
    "debezium.payload.op",
    "debezium.payload.source.lsn",
    "kafka_partition",
    "kafka_offset",
).show(1, truncate=False, vertical=True)

-RECORD 0------------------------------------------------------------------------------------------------------------------------------------
 before          | NULL                                                                                                                      
 after           | {2, Aarav, Sharma, aarav@example.com, NULL, 8633, ACTIVE, GOLD, 2026-08-11T11:27:09.474139Z, 2026-08-11T11:27:09.474139Z} 
 op              | r                                                                                                                         
 lsn             | 26967696                                                                                                                  
 kafka_partition | 0                                                                                                                         
 kafka_offset    | 0                                                                                                                         
only s

In [13]:
customer_cdc_data = customer_parsed_data.withColumn("customer",
                                                    F.when(
                                                        F.col("debezium.payload.op") == "d",
                                                        F.col("debezium.payload.before"),

                                                    ).otherwise(
                                                        F.col("debezium.payload.after")
                                                    )
                                                    )

In [14]:
customer_cdc_data_normalized = customer_cdc_data.select(
    F.col("customer.customer_id").alias("customer_id"),
    F.col("customer.first_name").alias("first_name"),
    F.col("customer.last_name").alias("last_name"),
    F.col("customer.email").alias("email"),
    F.col("customer.phone_number").alias("phone_number"),
    F.date_add(
        F.lit("1970-01-01").cast("date"),
        F.col("customer.date_of_birth").cast("int")
    ).alias("date_of_birth"),
    F.col("customer.status").alias("status"),
    F.col("customer.segment").alias("segment"),
    F.try_to_timestamp(F.col("customer.created_at")).alias("created_at"),
    F.try_to_timestamp(F.col("customer.updated_at")).alias("updated_at"),
    F.timestamp_millis(F.col("debezium.payload.ts_ms")).alias("cdc_timestamp"),
    F.timestamp_millis(F.col("debezium.payload.source.ts_ms")).alias("source_timestamp"),
    F.col("debezium.payload.op").alias("cdc_operation"),
    F.col("debezium.payload.source.lsn").alias("source_lsn"),
    F.col("debezium.payload.source.txId").alias("source_tx_id"),
    F.col("kafka_topic").alias("kafka_topic"),
    F.col("kafka_partition").alias("kafka_partition"),
    F.col("kafka_offset").alias("kafka_offset"),
    F.col("kafka_timestamp").alias("kafka_timestamp"),
    F.col("is_tombstone").alias("is_tombstone"),
    F.col("ingested_at").alias("ingested_at"),
)

In [15]:
customer_non_tombstone_data = customer_cdc_data_normalized.filter(
    ~F.col("is_tombstone")
)

In [16]:
customer_filter_condition = (
    F.array(
        F.when(F.col("customer_id").isNull(), F.lit("MISSING_CUSTOMER_ID")),
        F.when((F.col("first_name").isNull()| (F.trim(F.col("first_name")) == "")), F.lit("MISSING_FIRST_NAME")),
        F.when((F.col("last_name").isNull() |(F.trim(F.col("last_name")) == "")), F.lit("MISSING_LAST_NAME")),
        F.when((F.col("email").isNull() & F.col("phone_number").isNull()), F.lit("MISSING_CONTACT_INFO")),
        F.when(F.col("date_of_birth") > F.current_date(), F.lit("FUTURE_DATE_OF_BIRTH")),
        F.when(~F.col("status").isin(["ACTIVE","INACTIVE","SUSPENDED"]), F.lit("INVALID_STATUS")),
        F.when(~F.col("segment").isin(["STANDARD","GOLD","PREMIUM"]), F.lit("INVALID_SEGMENT"))
))

In [17]:
customer_filtered_data = customer_non_tombstone_data.withColumn("dq_errors", F.array_compact(customer_filter_condition))

In [18]:
customer_filtered_data.select(["customer_id","first_name","last_name","email","phone_number"
                                  ,"cdc_operation","is_tombstone","dq_errors" ]).show(truncate=False)

+-----------+----------+---------+------------------------+-------------+-------------+------------+---------+
|customer_id|first_name|last_name|email                   |phone_number |cdc_operation|is_tombstone|dq_errors|
+-----------+----------+---------+------------------------+-------------+-------------+------------+---------+
|2          |Aarav     |Sharma   |aarav@example.com       |NULL         |r            |false       |[]       |
|1          |Sailesh   |Naidu    |sailesh@example.com     |+919876543210|r            |false       |[]       |
|3          |Rahul     |Verma    |rahul.verma@example.com |+919999888877|c            |false       |[]       |
|3          |Rahul     |Verma    |rahul.verma@example.com |+919999888877|u            |false       |[]       |
|3          |Rahul     |Verma    |rahul.verma@example.com |+919999888877|u            |false       |[]       |
|3          |Rahul     |Verma    |rahul.verma@example.com |+919999888877|d            |false       |[]       |
|

In [19]:
(customer_filtered_data.select(["customer_id","first_name","last_name","email",
                               "phone_number","cdc_operation","is_tombstone","dq_errors" ])
 .show(truncate=False))

+-----------+----------+---------+------------------------+-------------+-------------+------------+---------+
|customer_id|first_name|last_name|email                   |phone_number |cdc_operation|is_tombstone|dq_errors|
+-----------+----------+---------+------------------------+-------------+-------------+------------+---------+
|2          |Aarav     |Sharma   |aarav@example.com       |NULL         |r            |false       |[]       |
|1          |Sailesh   |Naidu    |sailesh@example.com     |+919876543210|r            |false       |[]       |
|3          |Rahul     |Verma    |rahul.verma@example.com |+919999888877|c            |false       |[]       |
|3          |Rahul     |Verma    |rahul.verma@example.com |+919999888877|u            |false       |[]       |
|3          |Rahul     |Verma    |rahul.verma@example.com |+919999888877|u            |false       |[]       |
|3          |Rahul     |Verma    |rahul.verma@example.com |+919999888877|d            |false       |[]       |
|

In [20]:
customer_valid_data = customer_filtered_data.filter(F.size(F.col("dq_errors")) ==0)
customer_quarantine_data = customer_filtered_data.filter(F.size(F.col("dq_errors")) >0)

In [21]:
customer_quarantine_data = (customer_quarantine_data.withColumn("dq_error_count", F.size(F.col("dq_errors")))
                            .withColumn("quarantined_at", F.current_timestamp()))

In [22]:
# Persist DQ-invalid customer records for investigation
silver_customer_quarantine_path = paths.silver_path(
    "customer/cdc/customers/quarantine/notebook"
)


In [23]:
customer_quarantine_data.show(truncate=False, vertical=True)

(0 rows)


In [24]:
customer_valid_data = customer_valid_data.drop("dq_errors")

In [25]:
customer_deduplicated_data = customer_valid_data.drop_duplicates(["kafka_topic","kafka_partition", "kafka_offset"])

In [26]:
customer_deduplicated_data.select(
    "customer_id",
    "cdc_operation",
    "source_tx_id",
    "source_lsn",
    "kafka_partition",
    "kafka_offset",
).orderBy(
    "kafka_partition",
    "kafka_offset",
).show(truncate=False)


+-----------+-------------+------------+----------+---------------+------------+
|customer_id|cdc_operation|source_tx_id|source_lsn|kafka_partition|kafka_offset|
+-----------+-------------+------------+----------+---------------+------------+
|2          |r            |756         |26967696  |0              |0           |
|1          |r            |756         |26967696  |0              |1           |
|3          |c            |757         |26967896  |0              |2           |
|3          |u            |758         |26969856  |0              |3           |
|3          |u            |762         |26977544  |0              |4           |
|3          |d            |763         |26978152  |0              |5           |
|4          |c            |764         |26979704  |0              |7           |
|4          |u            |765         |26982496  |0              |8           |
|4          |d            |766         |26983096  |0              |9           |
|5          |c            |7

In [27]:
customer_deduplicated_data.count()

19

In [28]:
# S3 - CDC ordering, replay protection and persistent history

from delta.tables import DeltaTable
from pyspark.sql import Window


In [29]:
# Persistent Delta table containing observed CDC event history

silver_customer_history_path = paths.silver_path(
    "customer/cdc/customers/cdc_history/notebook"
)
# Check history before processing this run

customer_history_exists = DeltaTable.isDeltaTable(
    spark,
    silver_customer_history_path
)

In [37]:
if customer_history_exists:

    # Read PREVIOUS accepted history
    customer_cdc_history_table = DeltaTable.forPath(
        spark,
        silver_customer_history_path
    )

    customer_cdc_history_data = customer_cdc_history_table.toDF()

    # Latest accepted event for each customer
    customer_cdc_latest_window = Window.partitionBy(
        "customer_id"
    ).orderBy(
        F.col("source_lsn").desc(),
        F.col("kafka_offset").desc()
    )

    customer_cdc_latest = (
        customer_cdc_history_data
        .withColumn(
            "row_number",
            F.row_number().over(customer_cdc_latest_window)
        )
        .filter(F.col("row_number") == 1)
        .drop("row_number")
    )

    # Compare this run's incoming events against PREVIOUS history
    customer_cdc_comparison = (
        customer_deduplicated_data.alias("s")
        .join(
            customer_cdc_latest.alias("t"),
            F.col("s.customer_id") == F.col("t.customer_id"),
            "left"
        )
    )

    # Classify source ordering
    customer_cdc_classified = customer_cdc_comparison.withColumn(
        "cdc_status",

        F.when(
            F.col("t.customer_id").isNull(),
            F.lit("NEW")
        )
        .when(
            F.col("s.source_lsn") > F.col("t.source_lsn"),
            F.lit("NEWER")
        )
        .when(
            F.col("s.source_lsn") < F.col("t.source_lsn"),
            F.lit("STALE")
        )
        .when(
            F.col("s.kafka_partition") != F.col("t.kafka_partition"),
            F.lit("AMBIGUOUS_ORDERING")
        )
        .when(
            F.col("s.kafka_offset") > F.col("t.kafka_offset"),
            F.lit("NEWER")
        )
        .otherwise(
            F.lit("STALE")
        )
    )

    # Only accepted incoming events
    customer_cdc_accepted_events = (
        customer_cdc_classified
        .filter(F.col("cdc_status").isin("NEW", "NEWER"))
        .select("s.*")
    )

    # Keep these separately for later monitoring/quarantine
    customer_cdc_rejected_events = (
    customer_cdc_classified
    .filter(
        F.col("cdc_status").isin(
            "STALE",
            "AMBIGUOUS_ORDERING"
        )
    )
    .select(
        # Keep incoming event once
        "s.*",
        # Why we rejected it
        "cdc_status",
        # Previous accepted position it conflicted with
        F.col("t.source_lsn").alias("persisted_source_lsn"),
        F.col("t.kafka_partition").alias(
            "persisted_kafka_partition"
        ),
        F.col("t.kafka_offset").alias(
            "persisted_kafka_offset"
        )
    )
    .withColumn(
        "rejected_at",
        F.current_timestamp()
    )
)
else:
    # FIRST RUN:
    # There is no previous history to compare against.
    customer_cdc_accepted_events = customer_deduplicated_data

    customer_cdc_rejected_events = None

In [31]:
from delta.tables import DeltaTable
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


def merge_cdc_events(
    spark: SparkSession,
    data: DataFrame,
    target_path: str,
) -> None:
    """Persist CDC events idempotently using Kafka record identity."""

    if not DeltaTable.isDeltaTable(spark, target_path):
        (
            data.write
            .format("delta")
            .save(target_path)
        )
        return

    target_table = DeltaTable.forPath(
        spark,
        target_path
    )

    event_identity_condition = (
        (F.col("t.kafka_topic") == F.col("s.kafka_topic"))
        & (F.col("t.kafka_partition") == F.col("s.kafka_partition"))
        & (F.col("t.kafka_offset") == F.col("s.kafka_offset"))
    )

    (
        target_table.alias("t")
        .merge(
            data.alias("s"),
            event_identity_condition
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [32]:
silver_customer_cdc_rejected_path = paths.silver_path(
    "customer/cdc/customers/rejected/notebook"
    )
silver_customer_quarantine_path = paths.silver_path(
    "customer/cdc/customers/quarantine/notebook"
)

In [34]:
# DQ quarantine
merge_cdc_events(
    spark,
    customer_quarantine_data,
    silver_customer_quarantine_path,
)

# Accepted CDC history
merge_cdc_events(
    spark,
    customer_cdc_accepted_events,
    silver_customer_history_path,
)

# CDC ordering rejections only exist on normal runs
if customer_cdc_rejected_events is not None:
    merge_cdc_events(
        spark,
        customer_cdc_rejected_events,
        silver_customer_cdc_rejected_path,
    )

In [36]:
print(customer_cdc_rejected_events)

DataFrame[customer_id: bigint, first_name: string, last_name: string, email: string, phone_number: string, date_of_birth: date, status: string, segment: string, created_at: timestamp, updated_at: timestamp, cdc_timestamp: timestamp, source_timestamp: timestamp, cdc_operation: string, source_lsn: bigint, source_tx_id: bigint, kafka_topic: string, kafka_partition: int, kafka_offset: bigint, kafka_timestamp: timestamp, is_tombstone: boolean, ingested_at: timestamp, cdc_status: string, persisted_source_lsn: bigint, persisted_kafka_partition: int, persisted_kafka_offset: bigint, rejected_at: timestamp]
